# AgentGuard Model Training and Evaluation Notebook

This notebook implements the complete pipeline for training and evaluating the AgentGuard anomaly detection model, including:
- Fixed split training/validation/testing
- K-fold cross-validation at agent level
- Model inference and per-agent metrics computation

Instructions:
1. Update `config.yml` with your dataset and training parameters
2. Run cells sequentially from top to bottom
3. For **fixed split**: execute sections 1-5
4. For **k-fold CV**: execute sections 1-2, then 6
5. For **inference**: execute sections 1-2, then 7-8

## Section 1: Load Configuration and Dependencies
Import required libraries and load configuration parameters.

In [1]:
import argparse
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import numpy as np
from collections import defaultdict

from data.preprocessing import run_preprocessing
from data.dataset.telemetry_dataset import AgentGuardDataset
from data.dataset.collate import agentguard_collate
from models.agentguard import AgentGuardModel
from training.trainer import AgentGuardTrainer

In [2]:
import os
import random

def set_global_seed(seed: int = 42):
    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    print(f"Global seed set to {seed}")

# Set Global Seed
set_global_seed(42)

Global seed set to 42


In [3]:
def load_config(path="config.yml"):
    """Load configuration from YAML file."""
    with open(path) as f:
        return yaml.safe_load(f)

config = load_config("config.yml")
print("Configuration loaded successfully!\n")
print("=" * 60)
print("CONFIG PARAMETERS")
print("=" * 60)
for key, value in config.items():
    print(f"\n{key}:")
    if isinstance(value, dict):
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"  {value}")

Configuration loaded successfully!

CONFIG PARAMETERS

data:
  raw_data_dir: data/dataset/agentguard-all-batches
  processed_dir: data/processed
  date: 2026-03-15
  window_size: 30
  seq_context: 8
  max_seq_len: 64
  attacked_agents: ['agent-1', 'agent-2', 'agent-3', 'agent-4', 'agent-5', 'agent-6', 'agent-7', 'agent-8', 'agent-9', 'agent-10', 'agent-11', 'agent-12', 'agent-13', 'agent-14', 'agent-15']
  control_agents: ['agent-16', 'agent-17', 'agent-18', 'agent-19', 'agent-20']
  augmentation: none
  augmentation_prob: 0.0
  class_weight_ratio: 19.03
  k_folds: 5
  train_agents: ['agent-1', 'agent-2', 'agent-3', 'agent-6', 'agent-7', 'agent-9', 'agent-10', 'agent-11', 'agent-12', 'agent-16', 'agent-17', 'agent-18']
  val_agents: ['agent-4', 'agent-8', 'agent-13', 'agent-19']
  test_agents: ['agent-5', 'agent-14', 'agent-15', 'agent-20']

model:
  d_model: 128
  stream1_input_dim: 32
  stream2_input_dim: 28
  mamba_layers: 2
  transformer_layers: 4
  transformer_heads: 8
  transform

## Section 2: Build Model Architecture
Define and instantiate the AgentGuardModel with parameters from config.

In [4]:
def build_model(config):
    """Build AgentGuardModel from configuration."""
    model_cfg = config["model"]
    return AgentGuardModel(
        stream1_input_dim=model_cfg["stream1_input_dim"],
        stream2_input_dim=model_cfg["stream2_input_dim"],
        d_model=model_cfg["d_model"],
        latent_dim=model_cfg["latent_dim"],
        mamba_layers=model_cfg["mamba_layers"],
        mamba_state_dim=model_cfg["d_model"],
        transformer_layers=model_cfg["transformer_layers"],
        transformer_heads=model_cfg["transformer_heads"],
        transformer_ff_dim=model_cfg.get("transformer_ff_dim", 512),
        dropout=model_cfg["dropout"],
        max_seq_len=config["data"]["max_seq_len"],
        fusion_strategy=model_cfg.get("fusion_strategy", "cross_attention"),
        cls_head_layers=model_cfg.get("cls_head_layers", 2),
        cls_head_hidden_dim=model_cfg.get("cls_head_hidden_dim", 64),
        cls_head_activation=model_cfg.get("cls_head_activation", "relu"),
        decoder_activation=model_cfg.get("decoder_activation", "relu"),
    )

model = build_model(config)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model created successfully!")
print(f"Total trainable parameters: {num_params:,}")

Model created successfully!
Total trainable parameters: 1,777,825


## Section 3: Create Data Loaders
Define function to build train/validation/test DataLoaders from agent splits with proper normalization handling.

In [5]:
def build_loaders_from_splits(config, train_agents, val_agents, test_agents=None):
    """
    Build DataLoaders from explicit agent splits.
    
    Args:
        config: Configuration dictionary
        train_agents: List of agent IDs for training
        val_agents: List of agent IDs for validation
        test_agents: Optional list of agent IDs for testing
        
    Returns:
        Tuple of (train_loader, val_loader, test_loader)
    """
    data_cfg = config["data"]
    training_cfg = config["training"]

    # Build training dataset and compute normalization stats
    train_ds = AgentGuardDataset(
        data_cfg["processed_dir"], train_agents,
        seq_context=data_cfg["seq_context"], normalize=True,
    )
    train_mean, train_std = train_ds.get_normalization_stats()
    print(f"Train normalization stats computed from {len(train_agents)} agents")

    # Build validation dataset with training normalization
    val_ds = AgentGuardDataset(
        data_cfg["processed_dir"], val_agents,
        seq_context=data_cfg["seq_context"], normalize=False,
    )
    val_ds.set_normalization_stats(train_mean, train_std)

    # Create DataLoaders
    train_loader = DataLoader(
        train_ds, batch_size=training_cfg["batch_size"],
        shuffle=True, collate_fn=agentguard_collate,
    )
    val_loader = DataLoader(
        val_ds, batch_size=training_cfg["batch_size"],
        shuffle=False, collate_fn=agentguard_collate,
    )

    test_loader = None
    if test_agents:
        test_ds = AgentGuardDataset(
            data_cfg["processed_dir"], test_agents,
            seq_context=data_cfg["seq_context"], normalize=False,
        )
        test_ds.set_normalization_stats(train_mean, train_std)
        test_loader = DataLoader(
            test_ds, batch_size=training_cfg["batch_size"],
            shuffle=False, collate_fn=agentguard_collate,
        )
        print(f"Dataset sizes: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")
    else:
        print(f"Dataset sizes: train={len(train_ds)}, val={len(val_ds)}")

    return train_loader, val_loader, test_loader

## Section 4: Train Model with Fixed Split
Build data loaders and train the model using predefined train/validation/test splits.

In [6]:
# Step 1: Determine device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}\n")

# Step 2: Build fresh model
model = build_model(config)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}\n")

# Step 3: Build data loaders from config splits
train_agents = config["data"]["train_agents"]
val_agents = config["data"]["val_agents"]

print(f"Train agents: {train_agents}")
print(f"Val agents: {val_agents}\n")

train_loader, val_loader, _ = build_loaders_from_splits(config, train_agents, val_agents)
print("Data loaders created successfully!\n")

Using device: cuda

Model parameters: 1,777,825

Train agents: ['agent-1', 'agent-2', 'agent-3', 'agent-6', 'agent-7', 'agent-9', 'agent-10', 'agent-11', 'agent-12', 'agent-16', 'agent-17', 'agent-18']
Val agents: ['agent-4', 'agent-8', 'agent-13', 'agent-19']

Train normalization stats computed from 12 agents
Dataset sizes: train=7687, val=2565
Data loaders created successfully!



In [7]:
# Step 4: Set up logging (optional)
logger = epoch_csv = None
log_cfg = config.get("logging", {})

if log_cfg.get("enabled", False):
    from utils.logging import setup_run_logger, log_config, EpochCSVWriter
    logger, log_path = setup_run_logger("train", config, log_dir=log_cfg.get("log_dir", "logs"))
    log_config(logger, config)
    if log_cfg.get("save_epoch_csv", True):
        epoch_csv = EpochCSVWriter(log_path)
    print(f"Logging enabled. Logs will be saved to: {log_path}\n")
else:
    print("Logging disabled.\n")

Logging enabled. Logs will be saved to: logs\train_20260413_220600.log



In [8]:
# Step 5: Initialize trainer and fit
trainer = AgentGuardTrainer(
    model, train_loader, val_loader, config, device=device,
    logger=logger, epoch_csv=epoch_csv
)

print("Starting training...\n")
try:
    trainer.fit()
finally:
    if epoch_csv is not None:
        epoch_csv.close()
    print("\nTraining completed!")

Starting training...



c:\Users\zhaoh\Desktop\DL_Proj20\Theory-and-application-of-deep-learning\venv\lib\site-packages\torch\nn\modules\transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Epoch   1/150 | Train: 4.1328 (R=0.6632 C=3.4559 T=0.1366) | Val: 7.8559 | AUROC: 0.9616 AUPRC: 0.5707 F1: 0.2700 P: 0.1561 R: 1.0000 | LR: 0.000300
  -> Saved best model (F1=0.2700)
Epoch   2/150 | Train: 3.6314 (R=0.3187 C=3.2285 T=0.8412) | Val: 8.1156 | AUROC: 0.9934 AUPRC: 0.8501 F1: 0.5359 P: 0.3661 R: 1.0000 | LR: 0.000300
  -> Saved best model (F1=0.5359)
Epoch   3/150 | Train: 3.2856 (R=0.2267 C=2.9722 T=0.8674) | Val: 8.8278 | AUROC: 0.9962 AUPRC: 0.9098 F1: 0.8897 P: 0.8121 R: 0.9837 | LR: 0.000300
  -> Saved best model (F1=0.8897)
Epoch   4/150 | Train: 3.0619 (R=0.1638 C=2.8876 T=0.1054) | Val: 9.4988 | AUROC: 0.9965 AUPRC: 0.8843 F1: 0.9049 P: 0.8500 R: 0.9675 | LR: 0.000299
  -> Saved best model (F1=0.9049)
Epoch   5/150 | Train: 2.9806 (R=0.1311 C=2.8495 T=0.0000) | Val: 9.3582 | AUROC: 0.9968 AUPRC: 0.9162 F1: 0.7910 P: 0.6543 R: 1.0000 | LR: 0.000299
Epoch   6/150 | Train: 2.9147 (R=0.1186 C=2.7960 T=0.0014) | Val: 9.6348 | AUROC: 0.9949 AUPRC: 0.8565 F1: 0.8303 P: 0.

## Section 5: Evaluate Model on Test Set
Load trained model and evaluate on test set with comprehensive metrics.

In [9]:
# Step 1: Build test loader
test_agents = config["data"]["test_agents"]
print(f"Test agents: {test_agents}\n")

_, _, test_loader = build_loaders_from_splits(
    config,
    train_agents=config["data"]["train_agents"],
    val_agents=config["data"]["val_agents"],
    test_agents=test_agents,
)
print("Test loader created successfully!\n")

Test agents: ['agent-5', 'agent-14', 'agent-15', 'agent-20']

Train normalization stats computed from 12 agents
Dataset sizes: train=7687, val=2565, test=2546
Test loader created successfully!



In [10]:
# Step 2: Set up logging for evaluation
eval_logger = None
log_cfg = config.get("logging", {})
if log_cfg.get("enabled", False):
    from utils.logging import setup_run_logger, log_config
    eval_logger, _ = setup_run_logger("eval", config, log_dir=log_cfg.get("log_dir", "logs"))
    log_config(eval_logger, config)
    print("Logging enabled for evaluation.\n")

# Step 3: Evaluate on test set
eval_trainer = AgentGuardTrainer(model, None, None, config, device=device, logger=eval_logger)
print("Starting evaluation on test set...\n")

test_losses, test_metrics = eval_trainer.evaluate(test_loader)

print("\n" + "=" * 60)
print("TEST SET EVALUATION RESULTS")
print("=" * 60)
print("\nLosses:")
for loss_name, loss_val in test_losses.items():
    print(f"  {loss_name}: {loss_val:.4f}")

print("\nMetrics:")
for metric_name, metric_val in test_metrics.items():
    if isinstance(metric_val, float):
        print(f"  {metric_name}: {metric_val:.4f}")
    else:
        print(f"  {metric_name}: {metric_val}")

Logging enabled for evaluation.

Starting evaluation on test set...

Loaded best model from epoch 4

Test Results:
  Loss:  9.2135 (R=0.1276 C=3.1641 T=59.2179)
  AUROC:     0.9985
  AUPRC:     0.9634
  F1:        0.9187
  Precision: 0.8626
  Recall:    0.9826
  Samples: 2546 (anomalous=115, normal=2431)

TEST SET EVALUATION RESULTS

Losses:
  recon: 0.1276
  contrastive: 3.1641
  temporal: 59.2179
  total: 9.2135

Metrics:
  auroc: 0.9985
  auprc: 0.9634
  f1: 0.9187
  precision: 0.8626
  recall: 0.9826


In [11]:
torch.save(model.state_dict(), "model.pt")

## Section 6: Perform K-Fold Cross-Validation
Implement tier-aware stratified k-fold cross-validation at agent level with comprehensive result tracking.

In [12]:
def make_stratified_folds(attacked_agents, control_agents, k):
    """
    Create tier-aware stratified folds.
    
    Assumes attacked_agents ordered: 
    - Tier1 (indices 0-4): agents 1-5, ~100 anomalies each
    - Tier2 (indices 5-7): agents 6-8, ~21 anomalies each
    - Tier3 (indices 8+): agents 9-15, ~13 anomalies each
    
    Args:
        attacked_agents: List of attack agent IDs
        control_agents: List of control agent IDs
        k: Number of folds
        
    Returns:
        List of k folds, each containing agent IDs
    """
    tier1 = attacked_agents[:5]   # agents 1-5
    tier2 = attacked_agents[5:8]  # agents 6-8
    tier3 = attacked_agents[8:]   # agents 9-15

    folds = [[] for _ in range(k)]

    # Round-robin assignment within each tier independently
    for tier in [tier1, tier2, tier3, control_agents]:
        for i, agent in enumerate(tier):
            folds[i % k].append(agent)

    return folds

# Test fold creation
data_cfg = config["data"]
attacked = data_cfg["attacked_agents"]
control = data_cfg["control_agents"]
k = data_cfg["k_folds"]

test_folds = make_stratified_folds(attacked, control, k)
print(f"Stratified folds created:")
for fold_idx, fold_agents in enumerate(test_folds):
    print(f"  Fold {fold_idx + 1}: {fold_agents}")

Stratified folds created:
  Fold 1: ['agent-1', 'agent-6', 'agent-9', 'agent-14', 'agent-16']
  Fold 2: ['agent-2', 'agent-7', 'agent-10', 'agent-15', 'agent-17']
  Fold 3: ['agent-3', 'agent-8', 'agent-11', 'agent-18']
  Fold 4: ['agent-4', 'agent-12', 'agent-19']
  Fold 5: ['agent-5', 'agent-13', 'agent-20']


In [13]:
def do_cv(config):
    """
    Perform k-fold cross-validation at agent level.
    
    For each fold:
    1. Current fold agents -> test set
    2. Next fold agents -> validation set
    3. All other agents -> training set
    
    Each fold trains a fresh model.
    
    Args:
        config: Configuration dictionary
        
    Returns:
        List of result dictionaries, one per fold
    """
    data_cfg = config["data"]
    attacked = data_cfg["attacked_agents"]
    control = data_cfg["control_agents"]
    k = data_cfg["k_folds"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    print(f"Using device: {device}")
    print(f"Running {k}-fold stratified agent-level cross-validation")
    print(f"  Attacked agents: {len(attacked)}, Control agents: {len(control)}\n")

    folds = make_stratified_folds(attacked, control, k)

    # Set up logging for the entire CV run
    cv_logger = None
    log_cfg = config.get("logging", {})
    if log_cfg.get("enabled", False):
        from utils.logging import setup_run_logger, log_config
        cv_logger, _ = setup_run_logger("cv", config, log_dir=log_cfg.get("log_dir", "logs"))
        log_config(cv_logger, config)

    fold_results = []

    for fold_idx in range(k):
        print(f"\n{'='*60}")
        print(f"FOLD {fold_idx + 1}/{k}")
        print(f"{'='*60}")
        if cv_logger:
            cv_logger.info(f"\nFOLD {fold_idx + 1}/{k}")

        # Assign agents to folds
        test_agents = folds[fold_idx]
        val_agents = folds[(fold_idx + 1) % k]
        train_agents = []
        for j in range(k):
            if j != fold_idx and j != (fold_idx + 1) % k:
                train_agents.extend(folds[j])

        print(f"  Train: {train_agents}")
        print(f"  Val:   {val_agents}")
        print(f"  Test:  {test_agents}\n")

        # Fresh model per fold
        model = build_model(config)
        if fold_idx == 0:
            num_params = sum(p.numel() for p in model.parameters())
            print(f"Model parameters: {num_params:,}\n")

        # Build data loaders for this fold
        train_loader, val_loader, test_loader = build_loaders_from_splits(
            config, train_agents, val_agents, test_agents,
        )

        # Per-fold checkpoint
        checkpoint_dir = Path(data_cfg["processed_dir"]) / "checkpoints"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_path = checkpoint_dir / f"best_model_fold{fold_idx + 1}.pt"

        # Per-fold epoch CSV
        epoch_csv = None
        if log_cfg.get("enabled", False) and log_cfg.get("save_epoch_csv", True):
            from utils.logging import EpochCSVWriter, setup_run_logger
            fold_logger, fold_log_path = setup_run_logger(
                "cv", config, log_dir=log_cfg.get("log_dir", "logs"),
                trial_number=fold_idx + 1,
            )
            epoch_csv = EpochCSVWriter(fold_log_path)
        else:
            fold_logger = None

        # Train model for this fold
        trainer = AgentGuardTrainer(
            model, train_loader, val_loader, config,
            device=device, checkpoint_path=checkpoint_path,
            logger=fold_logger or cv_logger, epoch_csv=epoch_csv,
        )
        try:
            trainer.fit()
        finally:
            if epoch_csv is not None:
                epoch_csv.close()

        # Evaluate on test fold
        test_losses, test_metrics = trainer.evaluate(test_loader)
        fold_results.append({
            "fold": fold_idx + 1,
            "test_agents": test_agents,
            "val_agents": val_agents,
            "test_loss": test_losses["total"],
            "test_auroc": test_metrics["auroc"],
            "test_auprc": test_metrics["auprc"],
            "test_f1": test_metrics["f1"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "test_recon": test_losses["recon"],
            "test_contrastive": test_losses["contrastive"],
            "test_temporal": test_losses["temporal"],
        })

    # ── Summary ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"{k}-FOLD CROSS-VALIDATION SUMMARY")
    print(f"{'='*60}")

    aurocs = [r["test_auroc"] for r in fold_results]
    losses = [r["test_loss"] for r in fold_results]

    for r in fold_results:
        summary_line = (f"  Fold {r['fold']}: AUROC={r['test_auroc']:.4f}  "
                        f"Loss={r['test_loss']:.4f}  "
                        f"Test agents: {r['test_agents']}")
        print(summary_line)
        if cv_logger:
            cv_logger.info(summary_line)

    mean_auroc = sum(aurocs) / len(aurocs)
    std_auroc = (sum((a - mean_auroc) ** 2 for a in aurocs) / len(aurocs)) ** 0.5
    mean_loss = sum(losses) / len(losses)
    std_loss = (sum((l - mean_loss) ** 2 for l in losses) / len(losses)) ** 0.5

    print(f"\n  Mean AUROC: {mean_auroc:.4f} +/- {std_auroc:.4f}")
    print(f"  Mean Loss:  {mean_loss:.4f} +/- {std_loss:.4f}")
    print(f"{'='*60}")

    if cv_logger:
        cv_logger.info(f"\n  Mean AUROC: {mean_auroc:.4f} +/- {std_auroc:.4f}")
        cv_logger.info(f"  Mean Loss:  {mean_loss:.4f} +/- {std_loss:.4f}")
        cv_logger.info("=" * 60)

    return fold_results

In [14]:
# Run k-fold cross-validation
print("Starting k-fold cross-validation...\n")
cv_results = do_cv(config)

print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS SUMMARY")
print("=" * 60)
for result in cv_results:
    print(f"\nFold {result['fold']}:")
    print(f"  Test Agents: {result['test_agents']}")
    print(f"  AUROC: {result['test_auroc']:.4f}")
    print(f"  AUPRC: {result['test_auprc']:.4f}")
    print(f"  F1-Score: {result['test_f1']:.4f}")
    print(f"  Precision: {result['test_precision']:.4f}")
    print(f"  Recall: {result['test_recall']:.4f}")
    print(f"  Total Loss: {result['test_loss']:.4f}")

Starting k-fold cross-validation...

Using device: cuda
Running 5-fold stratified agent-level cross-validation
  Attacked agents: 15, Control agents: 5


FOLD 1/5
  Train: ['agent-3', 'agent-8', 'agent-11', 'agent-18', 'agent-4', 'agent-12', 'agent-19', 'agent-5', 'agent-13', 'agent-20']
  Val:   ['agent-2', 'agent-7', 'agent-10', 'agent-15', 'agent-17']
  Test:  ['agent-1', 'agent-6', 'agent-9', 'agent-14', 'agent-16']

Model parameters: 1,777,825

Train normalization stats computed from 10 agents
Dataset sizes: train=6560, val=3108, test=3130
Epoch   1/150 | Train: 4.2538 (R=0.6839 C=3.5202 T=0.4964) | Val: 7.0944 | AUROC: 0.8945 AUPRC: 0.4349 F1: 0.4604 P: 0.4571 R: 0.4638 | LR: 0.000300
  -> Saved best model (F1=0.4604)
Epoch   2/150 | Train: 3.5123 (R=0.3164 C=3.1854 T=0.1045) | Val: 6.6390 | AUROC: 0.8841 AUPRC: 0.2065 F1: 0.0000 P: 0.0000 R: 0.0000 | LR: 0.000300
Epoch   3/150 | Train: 3.3031 (R=0.2306 C=3.0332 T=0.3922) | Val: 8.6856 | AUROC: 0.9934 AUPRC: 0.8119 F1: 0.8243 P: 

## Section 7: Run Inference on Test Data
Load pretrained model and generate anomaly scores on test dataset.

In [15]:
def test_trained_model(test_config, weights_config):
    """
    Load trained model and run inference on test set.
    
    Args:
        test_config: Config with test data paths and inference parameters
        weights_config: Config with model architecture parameters
        
    Returns:
        Tuple of (anomaly_scores, true_labels, agent_ids)
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Build model with weights config architecture
    model = build_model(weights_config)

    # Load checkpoint
    ckpt_path = test_config["inference"]["checkpoint_path"]
    checkpoint = torch.load(ckpt_path, map_location=device)

    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model.to(device)
    model.eval()
    print(f"Loaded weights from: {ckpt_path}\n")

    data_cfg = test_config["data"]

    # Create test dataset
    test_ds = AgentGuardDataset(
        data_cfg["processed_dir"],
        data_cfg["test_agents"],
        seq_context=weights_config["data"]["seq_context"],
        normalize=False,
    )

    # Get normalization stats from training dataset
    train_agents = test_config["data"]["train_agents"]
    train_ds = AgentGuardDataset(
        data_cfg["processed_dir"],
        train_agents,
        seq_context=weights_config["data"]["seq_context"],
        normalize=True,
    )

    train_mean, train_std = train_ds.get_normalization_stats()
    test_ds.set_normalization_stats(train_mean, train_std)

    # Create test loader
    test_loader = DataLoader(
        test_ds,
        batch_size=test_config["inference"].get("batch_size", 32),
        shuffle=False,
        collate_fn=agentguard_collate,
    )

    print(f"Test dataset size: {len(test_ds)} samples")
    print(f"Test agents: {data_cfg['test_agents']}\n")

    # Run inference
    print("Running inference...\n")
    all_scores = []
    all_labels = []
    all_agent_ids = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            # Move tensors to device
            batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
            outputs = model(batch["stream1"], batch["stream2_seq"], batch["stream2_mask"])
            all_scores.append(outputs["anomaly_score"].squeeze(-1).cpu())
            all_labels.append(batch["label"].cpu())
            all_agent_ids.extend(batch["agent_id"])
            
            if (batch_idx + 1) % 10 == 0:
                print(f"  Processed batch {batch_idx + 1}/{len(test_loader)}")

    all_scores = torch.cat(all_scores).numpy()
    all_labels = torch.cat(all_labels).numpy()

    print(f"\nInference complete!")
    print(f"Total samples: {len(all_scores)}")
    print(f"Score range: [{all_scores.min():.4f}, {all_scores.max():.4f}]\n")

    return all_scores, all_labels, all_agent_ids

In [20]:
# Run inference (ensure checkpoint_path is set in config)
print("Starting inference on test set...\n")
test_config = load_config("test_config.yml")
scores, labels, agent_ids = test_trained_model(test_config, config)
print("Inference results obtained!")

Starting inference on test set...

Using device: cuda
Loaded weights from: data/processed/checkpoints/best_model_fold3.pt

Test dataset size: 4556 samples
Test agents: ['agent-4', 'agent-12', 'agent-19', 'agent-3', 'agent-8', 'agent-11', 'agent-18']

Running inference...

  Processed batch 10/143
  Processed batch 20/143
  Processed batch 30/143
  Processed batch 40/143
  Processed batch 50/143
  Processed batch 60/143
  Processed batch 70/143
  Processed batch 80/143
  Processed batch 90/143
  Processed batch 100/143
  Processed batch 110/143
  Processed batch 120/143
  Processed batch 130/143
  Processed batch 140/143

Inference complete!
Total samples: 4556
Score range: [0.3844, 0.6684]

Inference results obtained!


## Section 8: Compute Per-Agent Metrics
Calculate per-agent performance metrics with confusion matrices and binary classification metrics.

In [21]:
def evaluate_per_agent(scores, labels, agent_ids, threshold=0.5):
    """
    Compute classification metrics per agent.
    
    Args:
        scores: np.array of anomaly scores
        labels: np.array of true labels (0 = normal, 1 = anomaly)
        agent_ids: list of agent IDs
        threshold: Decision threshold for binary prediction
        
    Returns:
        Dictionary mapping agent_id -> metrics dict with:
            - num_samples, num_normal, num_anomaly
            - accuracy, precision, recall, f1
            - confusion_matrix
    """
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
    )
    
    # Convert scores to binary predictions
    preds = (scores >= threshold).astype(int)

    # Group by agent
    agent_data = defaultdict(lambda: {"labels": [], "preds": []})
    for agent, label, pred in zip(agent_ids, labels, preds):
        agent_data[agent]["labels"].append(label)
        agent_data[agent]["preds"].append(pred)

    # Compute metrics per agent
    agent_metrics = {}
    for agent, data in agent_data.items():
        y_true = np.array(data["labels"])
        y_pred = np.array(data["preds"])
        cm = confusion_matrix(y_true, y_pred)
        metrics = {
            "num_samples": len(y_true),
            "num_normal": int((y_true == 0).sum()),
            "num_anomaly": int((y_true == 1).sum()),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "confusion_matrix": cm,
        }
        agent_metrics[agent] = metrics

    return agent_metrics

In [22]:
# Compute per-agent metrics with threshold=0.5
threshold = 0.5
per_agent_metrics = evaluate_per_agent(scores, labels, agent_ids, threshold=threshold)

print(f"Per-Agent Metrics (threshold={threshold})")
print("=" * 80)

for agent, metrics in per_agent_metrics.items():
    print(f"\nAgent: {agent}")
    print(f"  Samples: {metrics['num_samples']} (Normal: {metrics['num_normal']}, Anomaly: {metrics['num_anomaly']})")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1-Score:  {metrics['f1']:.4f}")
    print(f"  Confusion Matrix:\n{metrics['confusion_matrix']}")

Per-Agent Metrics (threshold=0.5)

Agent: agent-4
  Samples: 946 (Normal: 853, Anomaly: 93)
  Accuracy:  0.9767
  Precision: 0.8381
  Recall:    0.9462
  F1-Score:  0.8889
  Confusion Matrix:
[[836  17]
 [  5  88]]

Agent: agent-12
  Samples: 543 (Normal: 532, Anomaly: 11)
  Accuracy:  0.9908
  Precision: 0.7143
  Recall:    0.9091
  F1-Score:  0.8000
  Confusion Matrix:
[[528   4]
 [  1  10]]

Agent: agent-19
  Samples: 506 (Normal: 506, Anomaly: 0)
  Accuracy:  0.9901
  Precision: 0.0000
  Recall:    0.0000
  F1-Score:  0.0000
  Confusion Matrix:
[[501   5]
 [  0   0]]

Agent: agent-3
  Samples: 945 (Normal: 852, Anomaly: 93)
  Accuracy:  0.9831
  Precision: 0.8532
  Recall:    1.0000
  F1-Score:  0.9208
  Confusion Matrix:
[[836  16]
 [  0  93]]

Agent: agent-8
  Samples: 576 (Normal: 557, Anomaly: 19)
  Accuracy:  0.9757
  Precision: 0.6000
  Recall:    0.7895
  F1-Score:  0.6818
  Confusion Matrix:
[[547  10]
 [  4  15]]

Agent: agent-11
  Samples: 534 (Normal: 524, Anomaly: 10)
 

In [23]:
# Compute overall metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

overall_preds = (scores >= threshold).astype(int)
overall_metrics = {
    "num_samples": len(labels),
    "num_normal": int((labels == 0).sum()),
    "num_anomaly": int((labels == 1).sum()),
    "accuracy": accuracy_score(labels, overall_preds),
    "precision": precision_score(labels, overall_preds, zero_division=0),
    "recall": recall_score(labels, overall_preds, zero_division=0),
    "f1": f1_score(labels, overall_preds, zero_division=0),
    "confusion_matrix": confusion_matrix(labels, overall_preds),
}

print("\n" + "=" * 80)
print("OVERALL METRICS")
print("=" * 80)
print(f"Total Samples: {overall_metrics['num_samples']}")
print(f"  Normal: {overall_metrics['num_normal']}")
print(f"  Anomaly: {overall_metrics['num_anomaly']}")
print(f"\nAccuracy:  {overall_metrics['accuracy']:.4f}")
print(f"Precision: {overall_metrics['precision']:.4f}")
print(f"Recall:    {overall_metrics['recall']:.4f}")
print(f"F1-Score:  {overall_metrics['f1']:.4f}")
print(f"\nConfusion Matrix:\n{overall_metrics['confusion_matrix']}")


OVERALL METRICS
Total Samples: 4556
  Normal: 4330
  Anomaly: 226

Accuracy:  0.9842
Precision: 0.7790
Recall:    0.9513
F1-Score:  0.8566

Confusion Matrix:
[[4269   61]
 [  11  215]]


## Summary

This notebook provides a complete training and evaluation pipeline for AgentGuard:

**Quick Start Guide:**
- **Fixed Split Training**: Run Sections 1-5 sequentially
- **K-Fold Cross-Validation**: Run Sections 1-2, then Section 6
- **Inference & Metrics**: Run Sections 1-2, then Sections 7-8

**Key Components:**
1. Configuration loading and model instantiation
2. Data loader creation with proper normalization
3. Training with optional logging
4. Evaluation metrics computation
5. K-fold cross-validation with stratified agent splits
6. Inference pipeline for anomaly detection
7. Per-agent and overall metrics computation

**Output Files:**
- Model checkpoints saved in `processed_dir/checkpoints/`
- Logs and epoch CSVs in `logs/` (if logging enabled)
- Test results in `test_results.md`